# Négatifs choisis à la main : cases de tôle saine sur grille

La greffe du carnet 02 a appris au modèle à dire « rien » (fausse alerte 80,2 % → 2,9 %, aire ROC
sur les 471 photos annotées 0,509 → 0,706). Elle reste sous le binaire dédié (0,855, fiche 0004),
et la cause identifiée est le **décalage de style** : positifs en gros plans CarDD nets (~1000 px),
négatifs en tuiles leboncoin tirées au hasard.

Inspection des tuiles aléatoires : **redondantes** (trois tirages dans la même photo se recouvrent)
et **à échelle incontrôlée** (une tuile de 448 px couvre ~45 % de la voiture — un plan large, pas un
gros plan).

**Ce carnet teste une autre fabrication des négatifs** : une grille 3×2 posée sur le détourage, et
un humain qui clique les cases montrant de la **tôle saine**. Chaque case devient un négatif au
format d'un gros plan, choisi plutôt que tiré au sort.

## Les quatre garde-fous

1. **Grille sur le détourage, pas sur la photo** : une case représente toujours la même fraction de
   véhicule, quelle que soit la distance de prise de vue.
2. **Appariement de netteté.** Photos leboncoin : 600×800 px typiques, détourage ~500×400, case
   ~165×200 px → **agrandie** vers 224. Images CarDD : ~1000 px → **réduites** vers 224. Sans
   correction, les intacts sont flous et les abîmés nets : le réseau apprendrait la netteté. On
   dégrade donc CarDD à la résolution effective des cases, et on **mesure avec et sans**.
3. **Anti-fuite** : annonces disjointes des 800 du lot d'annotation et des 697 de la greffe.
   Découpage train/val par annonce.
4. **Dose de fond testée, pas supposée** : n'entraîner que sur de la tôle prive le modèle de
   bitume, ciel et jantes, qu'il rencontrera pourtant à l'inférence. La variante C en réinjecte une
   part et on regarde l'effet.

In [ ]:
import sys
sys.path.insert(0, "../../src")

from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import roc_auc_score
from torch.utils.data import ConcatDataset, Subset
from torchvision import transforms

from cardd import CarDDMultiLabel, TuilesIntactes, CarDDAvecNegatifs, CLASSES
from device import get_device
import negatifs
import train_cardd as tc

COCO = "../../data/cardd/raw/CarDD_release/CarDD_COCO"
LOT = Path("../../data/cardd/negatifs_grille")
ALEA = Path("../../data/cardd/negatifs")
ANNOT = Path("../../data/leboncoin-private/annot")
GRAINE, TAILLE, LOTS, EPOQUES, COTE_EFFECTIF = 0, 224, 32, 10, 180

tc.set_seed(GRAINE)
device = get_device()
print("device :", device)

man_grille = pd.read_csv(LOT / "manifeste_grille.csv", dtype={"ad_id": str})
brut = pd.read_csv(LOT / "cases_saines.csv")
cases = brut[brut.case >= 0]                 # cases de tôle saine
vides = brut[brut.case < 0]                  # photos examinées, rien d'utilisable
print(f"{len(man_grille)} détourages proposés | {brut.fichier.nunique()} photos examinées")
print(f"{len(cases)} cases cliquées sur {cases.fichier.nunique()} photos "
      f"({len(cases)/max(cases.fichier.nunique(),1):.1f} par photo)")
print(f"{len(vides)} photos sans aucune case utilisable "
      f"({len(vides)/max(brut.fichier.nunique(),1)*100:.0f} % — intérieurs, moteurs, documents)")

# anti-fuite, vérifié avant toute découpe
exclus = set(pd.read_csv(ANNOT / "manifeste.csv", dtype={"ad_id": str}).ad_id)
exclus |= set(pd.read_csv(ALEA / "manifeste_negatifs.csv", dtype={"ad_id": str}).ad_id)
communs = set(man_grille.ad_id) & exclus
assert not communs, f"FUITE : {len(communs)} annonces déjà utilisées ailleurs"
print("anti-fuite : OK")

## 1. Découpe des cases cliquées

Les coordonnées ne viennent pas de la page : seul l'index de case a été exporté, et la découpe se
recalcule ici depuis la taille du détourage. Changer la grille ne demanderait pas de réannoter.

In [ ]:
man_cases = negatifs.decouper_cases(man_grille, cases, LOT, cols=3, lignes_grille=2,
                                    frac_val=0.2, seed=GRAINE)
print(man_cases.split.value_counts().to_string())
print(f"\ncôté médian des cases : {int(man_cases.cote_x.median())}x{int(man_cases.cote_y.median())} px")
print(f"annonces train/val disjointes : "
      f"{len(set(man_cases[man_cases.split=='train'].ad_id) & set(man_cases[man_cases.split=='val'].ad_id)) == 0}")

rng = np.random.default_rng(GRAINE)
choix = rng.choice(len(man_cases), min(12, len(man_cases)), replace=False)
fig, axes = plt.subplots(2, 6, figsize=(15, 5))
for ax, i in zip(axes.ravel(), choix):
    ax.imshow(Image.open(LOT / man_cases.iloc[i].fichier)); ax.axis("off")
    ax.set_title(f"case {man_cases.iloc[i].case}", fontsize=8)
plt.suptitle("Cases « tôle saine » choisies à la main — cible [0,0,0,0,0,0]")
plt.tight_layout(); plt.show()

## 2. L'appariement de netteté, vu à l'œil

À gauche une image CarDD telle quelle, à droite la même après passage à la résolution effective
d'une case leboncoin puis remontée. Si la différence saute aux yeux à 224 px, c'est qu'elle
sautait aussi aux « yeux » du réseau — et qu'il pouvait s'en servir pour trancher sans regarder le
dégât.

In [ ]:
ex = sorted(Path(f"{COCO}/train2017").glob("*.jpg"))[0]
brute = Image.open(ex).convert("RGB")
degradee = negatifs.degrader_comme_tuiles(brute, COTE_EFFECTIF)
case_ex = Image.open(LOT / man_cases.iloc[0].fichier)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, img, titre in zip(axes,
                          [brute.resize((224,224)), degradee.resize((224,224)),
                           case_ex.resize((224,224))],
                          [f"CarDD brute ({brute.size[0]}px) → 224",
                           f"CarDD dégradée (≈{COTE_EFFECTIF}px) → 224",
                           f"case leboncoin ({case_ex.size[0]}px) → 224"]):
    ax.imshow(img); ax.set_title(titre, fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

## 3. Les variantes

| variante | négatifs | netteté CarDD |
|---|---|---|
| référence | 1 671 tuiles aléatoires (carnet 02) | intacte |
| **A** | cases cliquées | intacte |
| **B** | cases cliquées | **appariée** |
| **C** | cases cliquées + dose de fond aléatoire | appariée |

Protocole identique à la fiche 0001 pour rester comparable : ResNet18 affinage complet, 224 px,
Adam 1e-4, lots de 32, 10 époques à budget fixe, meilleure époque sur validation, `test2017` fermé.

In [ ]:
tf_net = tc.build_transforms(TAILLE)
tf_apparie = transforms.Compose(
    [transforms.Lambda(lambda im: negatifs.degrader_comme_tuiles(im, COTE_EFFECTIF))]
    + list(tf_net.transforms))

def jeux(apparie: bool, avec_fond: bool):
    # (train, val) : CarDD + négatifs ; netteté appariée ou non, côté CarDD uniquement
    tf_cardd = tf_apparie if apparie else tf_net
    cardd_tr, cardd_va = (CarDDMultiLabel(COCO, s, tf_cardd) for s in ("train", "val"))
    neg_tr, neg_va = (TuilesIntactes(LOT / s, tf_net) for s in ("train", "val"))
    if avec_fond:
        alea_tr = TuilesIntactes(ALEA / "train", tf_net)
        alea_va = TuilesIntactes(ALEA / "val", tf_net)
        g = torch.Generator().manual_seed(GRAINE)
        pris_tr = torch.randperm(len(alea_tr), generator=g)[:len(neg_tr) // 3].tolist()
        pris_va = torch.randperm(len(alea_va), generator=g)[:len(neg_va) // 3].tolist()
        neg_tr = ConcatDataset([neg_tr, Subset(alea_tr, pris_tr)])
        neg_va = ConcatDataset([neg_va, Subset(alea_va, pris_va)])
    return (CarDDAvecNegatifs(cardd_tr, neg_tr), CarDDAvecNegatifs(cardd_va, neg_va),
            cardd_va, neg_va)

VARIANTES = [("A — cases, netteté brute", False, False),
             ("B — cases, netteté appariée", True, False),
             ("C — cases + fond, appariée", True, True)]

modeles, jeux_val = {}, {}
for nom, apparie, fond in VARIANTES:
    print(f"\n=== {nom} ===")
    tr, va, cardd_va, neg_va = jeux(apparie, fond)
    print(f"train : {len(tr)} images | val : {len(va)}")
    tc.set_seed(GRAINE)
    res = tc.train(tc.build_model("resnet18"), tr, va, device, epochs=EPOQUES, lr=1e-4,
                   batch_size=LOTS, num_workers=0, seed=GRAINE, verbose=False)
    m = tc.build_model("resnet18"); m.load_state_dict(res["best_state_dict"])
    modeles[nom] = m.to(device).eval()
    jeux_val[nom] = (cardd_va, neg_va)
    print(f"meilleure époque {res['meilleur_epoch']} | macro-F1 val combinée "
          f"{res['meilleur_macro_f1']:.4f} | {res['duree_epoque_s']:.0f} s/époque")

## 4. Verdict, volets 1 et 2 — typage préservé, et capacité à se taire

Typage mesuré sur `val2017` **seul** (comparable à 0,815 baseline et 0,818 greffe). Fausse alerte
mesurée sur les négatifs de validation **propres à chaque variante** — donc pas directement
comparable d'une ligne à l'autre, mais chacune face à sa propre difficulté.

In [ ]:
greffe, _ = tc.charger_checkpoint("../../models/cardd_greffe.pt", device)
cardd_val_net = CarDDMultiLabel(COCO, "val", tf_net)
loader_cardd = tc.make_loader(cardd_val_net, LOTS, num_workers=0)
cases_val = TuilesIntactes(LOT / "val", tf_net)
loader_cases = tc.make_loader(cases_val, LOTS, num_workers=0)

lignes = []
for nom, m in [("référence (greffe, carnet 02)", greffe)] + list(modeles.items()):
    p_cardd, cibles = tc.predict_probas(m, loader_cardd, device)
    s = tc.evaluate(p_cardd, cibles)
    p_cases, _ = tc.predict_probas(m, loader_cases, device)
    lignes.append({
        "modèle": nom,
        "macro-F1 val2017": round(float(s.loc["macro", "F1"]), 4),
        "fausse alerte / cases saines": f"{(p_cases.max(axis=1) > 0.5).mean()*100:.1f} %",
        "proba max moyenne": round(float(p_cases.max(axis=1).mean()), 3),
    })
verdict12 = pd.DataFrame(lignes)
display(verdict12)

## 5. Verdict, volet 3 — le juge de paix : les 471 photos annotées

Aire ROC contre les verdicts humains, en pleine photo (rapide, toutes les variantes) puis en
tuiles sur détourage pour la meilleure. Références : **0,509** (CarDD d'origine), **0,706**
(greffe), **0,855** (binaire dédié, fiche 0004).

In [ ]:
verdicts = pd.read_csv(ANNOT / "manifeste.csv", dtype={"id": str}).assign(
    fichier=lambda d: d.id + ".jpg").merge(pd.read_csv(ANNOT / "verdicts.csv"), on="fichier")
verdicts = verdicts[verdicts.verdict.isin(["intact", "abime"])].reset_index(drop=True)
y = (verdicts.verdict == "abime").astype(int).values
print(f"{len(verdicts)} photos ({y.sum()} abîmées)")

@torch.no_grad()
def probas_pleine_photo(m, dossier):
    out = []
    for f in verdicts.fichier:
        x = tf_net(Image.open(dossier / f).convert("RGB")).unsqueeze(0).to(device)
        out.append(torch.sigmoid(m(x))[0].cpu().numpy())
    return np.array(out)

roc = []
for nom, m in [("référence (greffe)", greffe)] + list(modeles.items()):
    p = probas_pleine_photo(m, ANNOT / "img")
    roc.append({"modèle": nom, "ROC pleine photo": round(roc_auc_score(y, p.max(axis=1)), 3)})
tab_roc = pd.DataFrame(roc)
display(tab_roc)

In [ ]:
from localize import tile_max_probs

meilleur = tab_roc.loc[tab_roc["ROC pleine photo"].idxmax(), "modèle"]
print("meilleure variante en pleine photo :", meilleur)

if meilleur != "référence (greffe)":
    m = modeles[meilleur]
    pt = np.array([[tile_max_probs(m, CLASSES, device,
                                   Image.open(ANNOT / "img_crop" / f).convert("RGB"))[0][c]
                    for c in CLASSES] for f in verdicts.fichier])
    roc_tuiles = roc_auc_score(y, pt.max(axis=1))
else:
    roc_tuiles = 0.706

display(pd.DataFrame([
    {"modèle": "CarDD d'origine (fiche 0002)", "ROC tuiles/détourage": 0.509},
    {"modèle": "greffe aléatoire (carnet 02)", "ROC tuiles/détourage": 0.706},
    {"modèle": meilleur, "ROC tuiles/détourage": round(float(roc_tuiles), 3)},
    {"modèle": "binaire dédié 384 px (fiche 0004)", "ROC tuiles/détourage": 0.855},
]))

## 6. Checkpoint

In [ ]:
if meilleur != "référence (greffe)":
    config = {"arch": "resnet18", "taille": TAILLE, "lots": LOTS, "epoques": EPOQUES, "lr": 1e-4,
              "graine": GRAINE, "variante": meilleur, "cote_effectif": COTE_EFFECTIF,
              "negatifs": "cases de tôle saine choisies à la main (grille 3x2)"}
    chemin = tc.sauver_checkpoint("../../models/cardd_greffe_grille.pt",
                                  {k: v.detach().cpu() for k, v in modeles[meilleur].state_dict().items()},
                                  config)
    print("écrit :", chemin)
else:
    print("aucune variante ne dépasse la référence : pas de nouveau checkpoint")

## 7. Conclusion

*(à rédiger après lecture : les cases choisies à la main battent-elles les tuiles aléatoires ?
l'appariement de netteté change-t-il quelque chose ? la dose de fond aide-t-elle ou nuit-elle ?)*

**Gate.** Arrêt ici. Fiches 0005 (greffe) et 0006 (négatifs choisis à la main) à écrire après
lecture.

Limites :
- les cases sont jugées par un seul annotateur, sur des photos de 600×800 px — un micro-dégât
  invisible à cette résolution passe pour de la tôle saine ;
- la fausse alerte de chaque variante est mesurée sur ses propres négatifs de validation : les
  colonnes ne se comparent pas entre elles, seulement à leur propre difficulté ;
- les 471 verdicts restent un lot unique, un seul segment, juillet 2026.